In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from tqdm import tqdm

%matplotlib inline

In [ ]:
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:csv_path = os.path.join(path, "Advertising.csv")
csv_path = os.path.join(path,'kaggle/input/q1-ka-ai-2026/Q1_data.csv.csv')

df = pd.read_csv(csv_path)
df = df.drop(columns="Unnamed: 0", axis=1)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here: to see the sekwed of the data and se if there is impalance
plt.figure(figsize=(8,5))
plt.hist(df["delivery_time"], bins=30)
plt.title("Target Distribution: delivery_time")
plt.xlabel("delivery_time (minutes)")
plt.ylabel("Count")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
if "Order_ID" in df.columns:
    df.drop(columns=["Order_ID"], inplace=True)

df.head()


In [ ]:
# Task 2: Write your code here:
# Separate target first to avoid leaking target filling logic
target_col = "delivery_time"

# Identify numeric and categorical columns (excluding target)
feature_cols = [c for c in df.columns if c != target_col]
num_cols = df[feature_cols].select_dtypes(include=["int64","float64"]).columns.tolist()
cat_cols = df[feature_cols].select_dtypes(include=["object","category","bool"]).columns.tolist()

# Fill missing values
for c in num_cols:
    df[c] = df[c].fillna(df[c].median())

for c in cat_cols:
    df[c] = df[c].fillna(df[c].mode()[0])

# If target has missing values, drop them (usually best for supervised tasks)
df = df.dropna(subset=[target_col])

df.isna().sum()


In [ ]:
# Task 3: Write your code here:
dup_count = df.duplicated().sum()
print("Duplicates:", dup_count)

if dup_count > 0:
    df = df.drop_duplicates()

print("After dropping duplicates:", df.duplicated().sum())


In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder


# One-hot encode categorical columns
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

df_encoded.head()


In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

X = df_encoded.drop(columns=[target_col])
y = df_encoded[target_col].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("X_scaled shape:", X_scaled.shape)


In [ ]:
# Task 6: Write your code here:
print("delivery_time stats:")
print(pd.Series(y).describe())

# Optional: see if there are extreme tails
plt.figure(figsize=(8,5))
plt.boxplot(y, vert=False)
plt.title("delivery_time Boxplot")
plt.xlabel("delivery_time")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Task 1: Write your code here:
# (Re-define cleanly just in case)
X = df_encoded.drop(columns=[target_col])
y = df_encoded[target_col].values


In [ ]:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_scaled), start=1):
    X_train, X_val = X_scaled[train_idx], X_scaled[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    model = RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_val)

    mae = mean_absolute_error(y_val, preds)
    mae_scores.append(mae)

    print(f"Fold {fold} MAE: {mae:.4f}")

print("\nAverage MAE across folds:", np.mean(mae_scores).round(4))


In [ ]:
# Task 1: Write your code here:
# Train final model on full data for interpretation
final_rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)
final_rf.fit(X_scaled, y)

importances = final_rf.feature_importances_
feat_names = X.columns

# Top 15
top_idx = np.argsort(importances)[-15:]

plt.figure(figsize=(10,6))
plt.barh(feat_names[top_idx], importances[top_idx])
plt.title("Top 15 Feature Importances (RandomForest)")
plt.xlabel("Importance")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Task 2: Write your code here:
pred_all = final_rf.predict(X_scaled)

plt.figure(figsize=(8,5))
plt.hist(pred_all, bins=30)
plt.title("Predicted Delivery Time Distribution")
plt.xlabel("Predicted delivery_time (minutes)")
plt.ylabel("Count")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Task Bonus: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# CatBoost (make sure it's installed in your environment)
from catboost import CatBoostRegressor

kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores_ens = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_scaled), start=1):
    X_train, X_val = X_scaled[train_idx], X_scaled[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    rf = RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )

    cb = CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=8,
        random_seed=42,
        verbose=0
    )

    rf.fit(X_train, y_train)
    cb.fit(X_train, y_train)

    pred_rf = rf.predict(X_val)
    pred_cb = cb.predict(X_val)

    pred_avg = (pred_rf + pred_cb) / 2.0

    mae = mean_absolute_error(y_val, pred_avg)
    mae_scores_ens.append(mae)

    print(f"Fold {fold} Ensemble MAE: {mae:.4f}")

print("\nAverage Ensemble MAE across folds:", np.mean(mae_scores_ens).round(4))
